# DeepEval — RAG Evaluation Practical

This notebook follows the **current DeepEval RAG evaluation API**.

We will:

1. Build a mini RAG application
2. Capture the actual generated answer and retrieved chunks
3. Create Goldens and `LLMTestCase`s
4. Evaluate the RAG system with the five core DeepEval RAG metrics:
   - Answer Relevancy
   - Faithfulness
   - Contextual Relevancy
   - Contextual Precision
   - Contextual Recall
5. Inspect metric scores + reasons
6. Build a full evaluation report
7. Export results
8. Use safe sequential settings to reduce rate-limit / timeout problems

## 0. Installation

In [ ]:
%pip install -U deepeval langchain langchain-openai pandas

## 1. DeepEval Safe Runtime Configuration

Run this **before importing DeepEval**.

These settings are intentionally conservative for notebooks/classes:

- Longer timeout budget
- Limited retries
- Later we run the main evaluation sequentially

This reduces the chance of the rate-limit and 180-second timeout errors that can happen when many LLM-as-a-judge calls run concurrently.

In [ ]:
import os

os.environ[
    "DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"
] = "240"

os.environ[
    "DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE"
] = "600"

os.environ[
    "DEEPEVAL_RETRY_MAX_ATTEMPTS"
] = "2"

print("DeepEval timeout/retry configuration set.")

## 2. Imports

In [ ]:
import getpass
import pandas as pd

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore

from deepeval import evaluate
from deepeval.dataset import EvaluationDataset, Golden
from deepeval.test_case import LLMTestCase
from deepeval.evaluate import AsyncConfig

from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    ContextualRelevancyMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
)

print("Imports successful.")

## 3. API Key + Models

We use:

- `RAG_MODEL` → generates the application answer
- `EMBEDDING_MODEL` → retrieves relevant chunks
- `DEEPEVAL_JUDGE_MODEL` → evaluates the RAG system

The judge model is intentionally configurable.

In [ ]:
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")

RAG_MODEL = os.getenv(
    "RAG_MODEL",
    "gpt-4.1-mini",
)

EMBEDDING_MODEL = os.getenv(
    "EMBEDDING_MODEL",
    "text-embedding-3-small",
)

DEEPEVAL_JUDGE_MODEL = os.getenv(
    "DEEPEVAL_JUDGE_MODEL",
    "gpt-4.1-mini",
)

print("RAG model:", RAG_MODEL)
print("Embedding model:", EMBEDDING_MODEL)
print("DeepEval judge model:", DEEPEVAL_JUDGE_MODEL)

# PART 1 — Build the RAG Application

## 4. Mini Knowledge Base

In [ ]:
documents = [
    Document(
        page_content=(
            "Full-time employees receive 24 paid leaves "
            "per calendar year."
        ),
        metadata={"doc_id": "leave_policy"},
    ),

    Document(
        page_content=(
            "Employees are allowed to work from home "
            "for a maximum of 2 days per week."
        ),
        metadata={"doc_id": "remote_policy"},
    ),

    Document(
        page_content=(
            "Employees can claim up to ₹3000 per month "
            "for internet reimbursement."
        ),
        metadata={"doc_id": "internet_policy"},
    ),

    Document(
        page_content=(
            "The standard probation period for new "
            "employees is 6 months."
        ),
        metadata={"doc_id": "probation_policy"},
    ),

    Document(
        page_content=(
            "Employees receive ₹1000 per month "
            "as mobile reimbursement."
        ),
        metadata={"doc_id": "mobile_policy"},
    ),

    Document(
        page_content=(
            "Medical insurance coverage begins from "
            "the employee's date of joining."
        ),
        metadata={"doc_id": "insurance_policy"},
    ),
]

print("Documents:", len(documents))
for doc in documents:
    print("-", doc.metadata["doc_id"])

## 5. Embeddings + Vector Store + Retriever

In [ ]:
embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL
)

vector_store = InMemoryVectorStore(
    embedding=embeddings
)

vector_store.add_documents(documents)

TOP_K = 3

retriever = vector_store.as_retriever(
    search_kwargs={"k": TOP_K}
)

print("Retriever ready. TOP_K =", TOP_K)

## 6. Generator LLM

In [ ]:
llm = ChatOpenAI(
    model=RAG_MODEL,
    temperature=0,
)

## 7. Complete RAG Pipeline

The RAG pipeline returns:

- `actual_output`
- `retrieval_context`
- `retrieved_doc_ids`

`retrieval_context` must contain the **actual chunks retrieved at runtime** because DeepEval uses them to evaluate retriever and generator quality.

In [ ]:
def rag_pipeline(query: str) -> dict:
    retrieved_docs = retriever.invoke(query)

    retrieval_context = [
        doc.page_content
        for doc in retrieved_docs
    ]

    retrieved_doc_ids = [
        doc.metadata.get("doc_id")
        for doc in retrieved_docs
    ]

    context = "\n\n".join(
        retrieval_context
    )

    prompt = f"""
You are an HR policy assistant.

Answer the user's question ONLY using the supplied context.

Rules:
1. Do not use outside knowledge.
2. Do not invent policy details.
3. If the answer is not available in the context, say:
   "I don't know based on the provided context."
4. Keep the answer concise.

CONTEXT:
{context}

QUESTION:
{query}
"""

    response = llm.invoke(prompt)

    return {
        "actual_output": response.content,
        "retrieval_context": retrieval_context,
        "retrieved_doc_ids": retrieved_doc_ids,
    }

## 8. Sanity Check the RAG Pipeline

In [ ]:
sample_query = "What is the monthly internet reimbursement limit?"

sample_result = rag_pipeline(sample_query)

print("QUESTION:")
print(sample_query)

print("\nACTUAL OUTPUT:")
print(sample_result["actual_output"])

print("\nRETRIEVED DOC IDS:")
print(sample_result["retrieved_doc_ids"])

print("\nRETRIEVAL CONTEXT:")

for i, chunk in enumerate(
    sample_result["retrieval_context"],
    start=1,
):
    print(f"\n--- Chunk {i} ---")
    print(chunk)

# PART 2 — Create the Evaluation Dataset

## 9. Goldens

A **Golden** is a pending evaluation example.

It contains the input and expected result, but it does not yet contain the runtime output produced by the RAG application.

Later:

```text
Golden
  ↓
Run RAG
  ↓
actual_output + retrieval_context
  ↓
LLMTestCase
```

In [ ]:
goldens = [
    Golden(
        input="How many paid leaves does a full-time employee receive?",
        expected_output=(
            "A full-time employee receives "
            "24 paid leaves per calendar year."
        ),
    ),

    Golden(
        input="How many work-from-home days are allowed per week?",
        expected_output=(
            "Employees can work from home "
            "for a maximum of 2 days per week."
        ),
    ),

    Golden(
        input="What is the monthly internet reimbursement limit?",
        expected_output=(
            "Employees can claim up to ₹3000 per month "
            "for internet reimbursement."
        ),
    ),

    Golden(
        input="What is the standard probation period for a new employee?",
        expected_output=(
            "The standard probation period "
            "for a new employee is 6 months."
        ),
    ),

    Golden(
        input="When does employee medical insurance coverage begin?",
        expected_output=(
            "Medical insurance coverage begins "
            "from the employee's date of joining."
        ),
    ),
]

dataset = EvaluationDataset(
    goldens=goldens
)

print("Goldens:", len(dataset.goldens))

## 10. Run the RAG Application and Create `LLMTestCase`s

For RAG evaluation, the important fields are:

| Field | Meaning |
|---|---|
| `input` | User's question |
| `actual_output` | RAG-generated answer |
| `expected_output` | Reference/ideal answer |
| `retrieval_context` | Actual chunks retrieved by the RAG pipeline |

In [ ]:
test_cases = []

rag_run_rows = []

for i, golden in enumerate(
    dataset.goldens,
    start=1,
):
    print(
        f"Running RAG {i}/{len(dataset.goldens)}"
    )

    result = rag_pipeline(
        golden.input
    )

    test_case = LLMTestCase(
        input=golden.input,
        actual_output=result["actual_output"],
        expected_output=golden.expected_output,
        retrieval_context=result["retrieval_context"],
    )

    test_cases.append(test_case)

    rag_run_rows.append({
        "input": golden.input,
        "expected_output": golden.expected_output,
        "actual_output": result["actual_output"],
        "retrieved_doc_ids": result["retrieved_doc_ids"],
    })

print("Test cases:", len(test_cases))

In [ ]:
pd.DataFrame(rag_run_rows)

# PART 3 — Configure the Five Core DeepEval RAG Metrics

## Generator metrics

### Answer Relevancy
Does the generated answer directly address the user query?

### Faithfulness
Are the answer's factual claims grounded in the retrieved context?

## Retriever metrics

### Contextual Relevancy
Are the retrieved chunks relevant/useful for the query?

### Contextual Precision
Are the most relevant chunks ranked higher?

### Contextual Recall
Did the retriever retrieve the information required for the ideal answer?

## 11. Metric Factory

We create fresh metric objects when needed.

For classroom/debugging reliability:

- `async_mode=False`
- explicit thresholds
- reasons enabled

In [ ]:
def create_rag_metrics():
    return [
        AnswerRelevancyMetric(
            threshold=0.80,
            model=DEEPEVAL_JUDGE_MODEL,
            include_reason=True,
            strict_mode=False,
            async_mode=False,
        ),

        FaithfulnessMetric(
            threshold=0.85,
            model=DEEPEVAL_JUDGE_MODEL,
            include_reason=True,
            strict_mode=False,
            async_mode=False,
        ),

        ContextualRelevancyMetric(
            threshold=0.75,
            model=DEEPEVAL_JUDGE_MODEL,
            include_reason=True,
            strict_mode=False,
            async_mode=False,
        ),

        ContextualPrecisionMetric(
            threshold=0.75,
            model=DEEPEVAL_JUDGE_MODEL,
            include_reason=True,
            strict_mode=False,
            async_mode=False,
        ),

        ContextualRecallMetric(
            threshold=0.80,
            model=DEEPEVAL_JUDGE_MODEL,
            include_reason=True,
            strict_mode=False,
            async_mode=False,
        ),
    ]


rag_metrics = create_rag_metrics()

print("Metrics:", len(rag_metrics))
for metric in rag_metrics:
    print("-", metric.__class__.__name__)

# PART 4 — Start Small: 1 Test Case × 1 Metric

Before running the entire evaluation suite, test the judge with the smallest possible workload.

This is useful for detecting:

- invalid API credentials
- provider rate limits
- judge-model availability problems
- network timeouts

In [ ]:
single_test_case = test_cases[0]

single_metric = AnswerRelevancyMetric(
    threshold=0.80,
    model=DEEPEVAL_JUDGE_MODEL,
    include_reason=True,
    async_mode=False,
)

single_metric.measure(
    single_test_case
)

print("Metric:", single_metric.__class__.__name__)
print("Score:", single_metric.score)
print("Threshold:", single_metric.threshold)
print("Passed:", single_metric.is_successful())
print("Reason:")
print(single_metric.reason)

# PART 5 — Debug One Test Case With All 5 Metrics

This is often the best way to teach and debug DeepEval because you can see exactly which metric is slow or failing.

In [ ]:
debug_case = test_cases[0]

debug_rows = []

for metric in create_rag_metrics():

    print("=" * 90)
    print("Running:", metric.__class__.__name__)

    metric.measure(
        debug_case
    )

    row = {
        "metric": metric.__class__.__name__,
        "score": metric.score,
        "threshold": metric.threshold,
        "passed": metric.is_successful(),
        "reason": metric.reason,
    }

    debug_rows.append(row)

    print("Score:", metric.score)
    print("Passed:", metric.is_successful())

debug_df = pd.DataFrame(debug_rows)
debug_df

## 12. Detailed Reasons

In [ ]:
for row in debug_rows:
    print("=" * 100)
    print("METRIC:", row["metric"])
    print("SCORE:", row["score"])
    print("THRESHOLD:", row["threshold"])
    print("PASSED:", row["passed"])
    print("REASON:")
    print(row["reason"])
    print()

# PART 6 — Full DeepEval Evaluation

We now run all test cases with all five metrics.

The evaluation is intentionally sequential:

```python
AsyncConfig(run_async=False)
```

This is slower, but safer for classroom/API-rate-limit environments.

In [ ]:
rag_metrics = create_rag_metrics()

evaluation_result = evaluate(
    test_cases=test_cases,
    metrics=rag_metrics,

    async_config=AsyncConfig(
        run_async=False
    ),

    hyperparameters={
        "rag_model": RAG_MODEL,
        "embedding_model": EMBEDDING_MODEL,
        "judge_model": DEEPEVAL_JUDGE_MODEL,
        "top_k": TOP_K,
    },
)

evaluation_result

# PART 7 — Build a Custom Detailed DataFrame

DeepEval's built-in evaluation result is useful, but for teaching and experimentation we may also want a simple tabular report.

To keep the execution easy to debug, the following code evaluates each test case and metric sequentially.

In [ ]:
detailed_results = []

for case_index, test_case in enumerate(
    test_cases,
    start=1,
):
    print(
        f"Evaluating test case "
        f"{case_index}/{len(test_cases)}"
    )

    row = {
        "input": test_case.input,
        "actual_output": test_case.actual_output,
        "expected_output": test_case.expected_output,
    }

    for metric in create_rag_metrics():

        metric.measure(
            test_case
        )

        metric_name = (
            metric.__class__.__name__
        )

        row[
            f"{metric_name}_score"
        ] = metric.score

        row[
            f"{metric_name}_passed"
        ] = metric.is_successful()

        row[
            f"{metric_name}_reason"
        ] = metric.reason

    detailed_results.append(
        row
    )

detailed_df = pd.DataFrame(
    detailed_results
)

detailed_df

## 13. Compact Score Table

In [ ]:
score_columns = [
    "input",
    "AnswerRelevancyMetric_score",
    "FaithfulnessMetric_score",
    "ContextualRelevancyMetric_score",
    "ContextualPrecisionMetric_score",
    "ContextualRecallMetric_score",
]

detailed_df[
    score_columns
]

## 14. Aggregate Metric Report

In [ ]:
metric_score_columns = [
    "AnswerRelevancyMetric_score",
    "FaithfulnessMetric_score",
    "ContextualRelevancyMetric_score",
    "ContextualPrecisionMetric_score",
    "ContextualRecallMetric_score",
]

aggregate_report = (
    detailed_df[
        metric_score_columns
    ]
    .mean()
    .rename({
        "AnswerRelevancyMetric_score":
            "answer_relevancy",

        "FaithfulnessMetric_score":
            "faithfulness",

        "ContextualRelevancyMetric_score":
            "contextual_relevancy",

        "ContextualPrecisionMetric_score":
            "contextual_precision",

        "ContextualRecallMetric_score":
            "contextual_recall",
    })
    .to_frame(
        name="mean_score"
    )
)

aggregate_report

## 15. Pass-Rate Report

In [ ]:
pass_columns = {
    "Answer Relevancy":
        "AnswerRelevancyMetric_passed",

    "Faithfulness":
        "FaithfulnessMetric_passed",

    "Contextual Relevancy":
        "ContextualRelevancyMetric_passed",

    "Contextual Precision":
        "ContextualPrecisionMetric_passed",

    "Contextual Recall":
        "ContextualRecallMetric_passed",
}


pass_report = pd.DataFrame([
    {
        "metric": metric_name,
        "pass_rate": float(
            detailed_df[
                column_name
            ].mean()
        ),
    }
    for metric_name, column_name
    in pass_columns.items()
])

pass_report

# PART 8 — Diagnose RAG Failures

This helper is not a DeepEval built-in metric. It simply maps DeepEval scores to likely engineering areas to inspect.

In [ ]:
def diagnose_rag(row):
    issues = []

    if (
        row[
            "ContextualRelevancyMetric_score"
        ] < 0.75
    ):
        issues.append(
            "Retrieved contexts contain too much irrelevant/noisy information."
        )

    if (
        row[
            "ContextualPrecisionMetric_score"
        ] < 0.75
    ):
        issues.append(
            "Relevant chunks may not be ranked high enough."
        )

    if (
        row[
            "ContextualRecallMetric_score"
        ] < 0.80
    ):
        issues.append(
            "Retriever may be missing information required for the ideal answer."
        )

    if (
        row[
            "FaithfulnessMetric_score"
        ] < 0.85
    ):
        issues.append(
            "Generator may be making claims unsupported by retrieved context."
        )

    if (
        row[
            "AnswerRelevancyMetric_score"
        ] < 0.80
    ):
        issues.append(
            "Generated answer may not directly address the user's question."
        )

    if not issues:
        return "No obvious issue based on the configured thresholds."

    return " | ".join(issues)


diagnostic_df = detailed_df.copy()

diagnostic_df[
    "diagnosis"
] = diagnostic_df.apply(
    diagnose_rag,
    axis=1,
)

diagnostic_df[
    [
        "input",
        "AnswerRelevancyMetric_score",
        "FaithfulnessMetric_score",
        "ContextualRelevancyMetric_score",
        "ContextualPrecisionMetric_score",
        "ContextualRecallMetric_score",
        "diagnosis",
    ]
]

# PART 9 — Export Results

In [ ]:
detailed_df.to_csv(
    "deepeval_rag_detailed_results.csv",
    index=False,
)

aggregate_report.to_csv(
    "deepeval_rag_aggregate_report.csv"
)

pass_report.to_csv(
    "deepeval_rag_pass_report.csv",
    index=False,
)

diagnostic_df.to_csv(
    "deepeval_rag_diagnostics.csv",
    index=False,
)

print("Saved:")
print("- deepeval_rag_detailed_results.csv")
print("- deepeval_rag_aggregate_report.csv")
print("- deepeval_rag_pass_report.csv")
print("- deepeval_rag_diagnostics.csv")

# PART 10 — Use DeepEval With Your Existing RAG

Your own RAG function only needs to provide:

```python
{
    "actual_output": "...",
    "retrieval_context": [
        "actual retrieved chunk 1",
        "actual retrieved chunk 2",
        ...
    ]
}
```

Then construct:

```python
LLMTestCase(
    input=query,
    actual_output=result["actual_output"],
    expected_output=reference_answer,
    retrieval_context=result["retrieval_context"],
)
```

and run the same metrics.

# Metric Mental Model

```text
                         RAG
                          │
              ┌───────────┴───────────┐
              │                       │
          RETRIEVER                GENERATOR
              │                       │
              │                 Answer Relevancy
              │                 Faithfulness
              │
      Contextual Relevancy
      Contextual Precision
      Contextual Recall
```

| Metric | Main Question | Component |
|---|---|---|
| Answer Relevancy | Does the answer address the query? | Generator |
| Faithfulness | Is the answer grounded in retrieved evidence? | Generator |
| Contextual Relevancy | Are retrieved chunks relevant/useful? | Retriever |
| Contextual Precision | Are useful chunks ranked high? | Retriever / Reranker |
| Contextual Recall | Was all information needed for the ideal answer retrieved? | Retriever |

# Important Runtime Note

DeepEval judge-based metrics are **not simple local mathematical formulas**.

They can invoke an evaluation model multiple times internally.

Therefore:

```text
5 test cases
×
5 metrics
≠ necessarily only 25 API requests
```

For debugging, always start with:

```text
1 test case
×
1 metric
```

Then:

```text
1 test case
×
5 metrics
```

Then run the full dataset.

If your provider has enough capacity and you want faster evaluation later, you can enable async execution with a small `max_concurrent` and throttle value.

# Final Architecture

```text
                    GOLDEN DATASET
                         │
                         ▼
                       Input
                         │
                         ▼
                        RAG
              ┌──────────┴──────────┐
              │                     │
          Retriever              Generator
              │                     │
              ▼                     ▼
      retrieval_context       actual_output
              │                     │
              └──────────┬──────────┘
                         │
                  expected_output
                         │
                         ▼
                    LLMTestCase
                         │
                         ▼
                      DeepEval
                         │
       ┌─────────────────┼─────────────────┐
       │                 │                 │
  Retriever Evals   Generator Evals   Pass / Fail
       │                 │
Contextual Rel.      Answer Rel.
Contextual Prec.     Faithfulness
Contextual Recall
                         │
                         ▼
                  Evaluation Report
```